Copyright Matlantis Corp. as contributors to Matlantis contrib project

# MBAR法による自由エネルギー計算（追加サンプリングを含む）

このノートブックは `06_mbar_free_energy_ja.ipynb` の拡張版です。
`07_umbrella_sampling_extra_near_rc12_ja.ipynb` で RC≈12 Å 付近に追加した強拘束ウインドウ
(`KAPPA=10.0 eV`) を、元のウインドウ(`KAPPA=2.5 eV`) と一緒に MBAR で解析します。

MBAR (Multistate Bennett Acceptance Ratio) は、ウインドウごとに異なるバイアス（バネ定数・拘束中心）を
正しく扱えるため、バネ定数が異なるウインドウを混在させても厳密に統合できます。

## Step 0. ライブラリのインポート

In [ ]:
# 必要に応じてmbarパッケージをインストールする
#! pip install pymbar

In [ ]:
# Reference
#   https://weitsehsu.com/course/enhanced_sampling/us/
#   https://github.com/choderalab/pymbar-examples/blob/master/umbrella-sampling-pmf/umbrella-sampling.py

import os
import glob

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap

import pymbar
from pymbar import timeseries

%config InlineBackend.figure_format = "retina"

## Step 1. 定数の設定

温度は元の計算・追加計算で共通（375 K）です。逆温度 $\beta=1/(k_BT)$ を計算しておきます。

In [ ]:
# Step1: Settings
kB   = 8.617333262e-5  # Boltzmann constant in eV/K
T    = 375.0           # single temperature (Kelvin) : 元・追加とも共通
beta = 1 / (kB * T)
print(f"beta = {beta:.4f} 1/eV")

## Step 2. 解析対象ウィンドウの定義（元 + 追加）

元のウインドウと追加のウインドウを、それぞれ 中心・バネ定数・出力ディレクトリをセットで登録します。
`build_window_list` は各出力ディレクトリを走査して `(center, kappa, dir)` のリストを作ります。

- 元のウインドウ: `KAPPA=2.5 eV`
- 追加のウインドウ: `KAPPA=10.0 eV`

**note**: 元のウインドウは `output/05_umbrella_sampling/` にフォルダがない場合、`assets/05_umbrella_sampling/` から読み込みます。追加のウインドウは 07 の出力（`output/07_umbrella_sampling_extra/`）が必要です。
  （`meta.txt` があればそこに記録された `kappa` を優先して読み取ります）

In [ ]:
# ============================================================
# 2. Window definition (original + extra)
# ============================================================
KAPPA_ORIG  = 2.5    # eV : 元のUSウインドウ
KAPPA_EXTRA = 10.0   # eV : 追加USウインドウ (07で使用した値。異なる場合は合わせる)

# outputから読み込み、なければassetsから読み込む
ORIG_DIR  = "./output/05_umbrella_sampling"
if not os.path.isdir(ORIG_DIR):
    ORIG_DIR = "./assets/05_umbrella_sampling"
    print(f"outputにフォルダが見つからないため、assetsから読み込みます: {ORIG_DIR}")
EXTRA_DIR = "./output/07_umbrella_sampling_extra"  # 07 の出力 (assets には用意していない)

def build_window_list(base_dir, default_kappa):
    """base_dir 以下の cv_at_* を走査し、(center, kappa, dir) のリストを返す。
    meta.txt に kappa 記載があればそれを優先する。"""
    windows = []
    for d in sorted(glob.glob(f"{base_dir}/cv_at_*")):
        center = float(os.path.basename(d).replace("cv_at_", ""))
        colvar = f"{d}/COLVAR_{center:.2f}"
        if not os.path.exists(colvar):
            continue  # 未計算のウインドウはスキップ
        kappa = default_kappa
        meta = f"{d}/meta.txt"
        if os.path.exists(meta):
            for line in open(meta):
                if line.startswith("kappa"):
                    kappa = float(line.split()[1])
        windows.append({"center": center, "kappa": kappa, "dir": d})
    return windows

windows = build_window_list(ORIG_DIR, KAPPA_ORIG) + build_window_list(EXTRA_DIR, KAPPA_EXTRA)
# 中心でソート (見やすさのため。MBARの結果は順序に依存しない)
windows = sorted(windows, key=lambda w: (w["center"], w["kappa"]))

print(f"Total windows: {len(windows)}")
print(f"  original (kappa={KAPPA_ORIG}) : {sum(1 for w in windows if w['kappa']==KAPPA_ORIG)}")
print(f"  extra    (kappa={KAPPA_EXTRA}): {sum(1 for w in windows if w['kappa']==KAPPA_EXTRA)}")
for w in windows:
    print(f"  center={w['center']:6.2f}  kappa={w['kappa']:5.1f}  {w['dir']}")

## Step 3. データの読み込み

各ウインドウのディレクトリから以下を読み込みます。

- `COLVAR_*` : 反応座標（z 位置）とバイアスの時系列
- `md-dyn.log` : ポテンシャルエネルギーの時系列

反応座標とポテンシャルの時刻を揃えるため、両者とも先頭（t=0）を除いて格納します
（`06` にあった長さ不一致を解消しています）。ウインドウごとにサンプル数が異なっても扱えるよう、
ここでは Python のリストに格納します。

In [ ]:
# ============================================================
# 3. Load trajectory data (variable-length safe)
# ============================================================
z_list = []   # 反応座標
w_list = []   # バイアスポテンシャル (参考用)
U_list = []   # ポテンシャルエネルギー (バイアス無し)
K_k    = []   # ウインドウごとのバネ定数
z0_k   = []   # ウインドウごとの拘束中心
beta_k = []   # ウインドウごとの逆温度

for w in windows:
    center, kappa, d = w["center"], w["kappa"], w["dir"]

    colvar = np.loadtxt(f"{d}/COLVAR_{center:.2f}")

    with open(f"{d}/md-dyn.log") as f:
        log_lines = [line for line in f if "Time" not in line]
    pot = np.loadtxt(log_lines)[:, 2]  # potential energy [eV]

    # 反応座標とポテンシャルの時刻を揃える (どちらも t=0 を除外)
    n = min(colvar.shape[0], pot.shape[0]) - 1  # 共通長 (先頭除外)
    z_list.append(colvar[1:1 + n, 1])
    w_list.append(colvar[1:1 + n, 2])
    U_list.append(pot[1:1 + n])

    K_k.append(kappa)
    z0_k.append(center)
    beta_k.append(beta)

K_k    = np.array(K_k)
z0_k   = np.array(z0_k)
beta_k = np.array(beta_k)

print(f"windows={len(windows)}, sample lengths: "
      f"min={min(len(z) for z in z_list)}, max={max(len(z) for z in z_list)}")

## Step 4. データの前処理 (Subsampling)

MD データは時系列相関があるため、`pymbar.timeseries` で統計的非効率性を評価し、
相関を除いた独立サンプルのみを抽出します。ウインドウごとにサンプル数 `N_k` が変わります。

In [ ]:
# ============================================================
# 4. Subsampling (decorrelation)
# ============================================================
K     = len(windows)                       # ウインドウ数
N_max = max(len(z) for z in z_list)         # 最大スナップショット数

N_k  = np.zeros([K], np.int32)
g_k  = np.zeros([K], np.float64)
z_kn = np.zeros([K, N_max], np.float64)
U_kn = np.zeros([K, N_max], np.float64)

for k in range(K):
    z_full = z_list[k]
    U_full = U_list[k]

    g_k[k]  = timeseries.statistical_inefficiency(z_full)
    indices = timeseries.subsample_correlated_data(z_full, g=g_k[k])
    N_k[k]  = len(indices)

    z_kn[k, :N_k[k]] = z_full[indices]
    U_kn[k, :N_k[k]] = U_full[indices]

print("Statistical inefficiency g_k (min/mean/max):",
      f"{g_k.min():.1f} / {g_k.mean():.1f} / {g_k.max():.1f}")
print("Independent samples N_k (min/mean/max):",
      f"{N_k.min()} / {N_k.mean():.0f} / {N_k.max()}")

## Step 5. Reduced Potential Matrix の計算

ウインドウ $k$ のスナップショット $n$ を、別のウインドウ $l$ のバイアス下で評価したときの被約ポテンシャル

$$u_{k,l,n} = \beta\left[U(x_{kn}) + \tfrac{1}{2}\kappa_l (z_{kn}-z^0_l)^2\right]$$

を計算します。ウインドウごとに **異なる $\kappa_l$** を使う点がポイントです。

In [ ]:
# ============================================================
# 5. Reduced potential matrix
# ============================================================
# PMF プロット用のビン設定
z_min = 10.0
z_max = 16.0
nbins = 60   # 追加ウインドウで分解能が上がったため元(40)より細かく

bin_edges    = np.linspace(z_min, z_max, nbins + 1)
bin_center_i = 0.5 * (bin_edges[:-1] + bin_edges[1:])

# u_kln[k,l,n] : ウインドウkのスナップショットn を ウインドウl のバイアスで評価した被約ポテンシャル
N_max = int(np.max(N_k))
u_kln = np.zeros([K, K, N_max])

for k in range(K):
    for n in range(N_k[k]):
        # バイアス無しの被約ポテンシャル
        base = beta_k[k] * U_kn[k, n]
        # 各ウインドウ l のバイアス (ウインドウごとの kappa_l, z0_l)
        u_kln[k, :, n] = base + beta_k[k] * 0.5 * K_k * (z_kn[k, n] - z0_k) ** 2

# バイアス無し被約ポテンシャル (FES 評価用)
u_kn = np.zeros([K, N_max])
for k in range(K):
    u_kn[k, :N_k[k]] = beta_k[k] * U_kn[k, :N_k[k]]

print("u_kln shape:", u_kln.shape)

## Step 6. MBAR による自由エネルギープロファイルの計算

`pymbar.FES` で再重み付けを行い、バイアスを除いた PMF をヒストグラム形式で求めます。

In [ ]:
# ============================================================
# 6. Compute FES with MBAR
# ============================================================
fes = pymbar.FES(u_kln, N_k, verbose=True)
z_n = pymbar.utils.kn_to_n(z_kn, N_k=N_k)

histo_params = {"bin_edges": bin_edges}
fes.generate_fes(u_kn, z_n, fes_type="histogram", histogram_parameters=histo_params)
results = fes.get_fes(bin_center_i, reference_point="from-lowest",
                      uncertainty_method="analytical")

f_i  = results["f_i"]
df_i = results["df_i"]

out_dir = "./output/08_mbar_free_energy_with_extra"
os.makedirs(out_dir, exist_ok=True)
out_path = f"{out_dir}/fes_with_extra.dat"
with open(out_path, "w") as f:
    f.write("# free energy profile (in units of kT), original + extra windows\n")
    f.write(f"# {'bin':>8s} {'f':>8s} {'df':>8s} \n")
    for i in range(len(bin_center_i)):
        f.write(f"{bin_center_i[i]:>8.3f} {f_i[i]:>8.3f} {df_i[i]:>8.3f} \n")
print("Saved:", out_path)

## Step 7. 結果の可視化（追加前 vs 追加後）

追加サンプリングを含む PMF をプロットします。元の `06` の結果 が
残っていれば重ねて描画します。

In [ ]:
# ============================================================
# 7. Plot: before vs after additional sampling
# ============================================================
data_new = np.loadtxt(out_path)
zc_new = data_new[:, 0]
f_new  = data_new[:, 1] - data_new[:, 1].min()
df_new = data_new[:, 2]

fig, ax = plt.subplots(figsize=(7, 5))

# 追加前 (あれば)
fes_orig = "./output/06_mbar_free_energy/fes.dat"
if os.path.exists(fes_orig):
    data_old = np.loadtxt(fes_orig)
    zc_old = data_old[:, 0]
    f_old  = data_old[:, 1] - data_old[:, 1].min()
    df_old = data_old[:, 2]
    ax.plot(zc_old, f_old, color="#8d99ae", linewidth=2.0,
            linestyle="--", label="Original only (06)")
    ax.fill_between(zc_old, f_old - df_old, f_old + df_old,
                    color="#8d99ae", alpha=0.20)

# 追加後
ax.plot(zc_new, f_new, color="#023e8a", linewidth=2.5,
        label="Original + extra (08)")
ax.fill_between(zc_new, f_new - df_new, f_new + df_new,
                color="#ade8f4", alpha=0.8)

# 追加サンプリングした領域をハイライト
ax.axvspan(11.6, 12.4, color="#ffd166", alpha=0.15, label="Extra sampling region")

ax.set_xlim(10, 16)
ax.set_ylim(0, 100)
ax.set_xlabel("Z position (A)", fontsize=14)
ax.set_ylabel("Free energy (kT)", fontsize=14)
ax.tick_params(labelsize=12)
ax.legend(fontsize=11)
ax.grid(True)

plt.savefig(f"{out_dir}/mbar_free_energy_with_extra.png", bbox_inches="tight")
plt.savefig(f"{out_dir}/mbar_free_energy_with_extra.pdf", bbox_inches="tight")
plt.show()

## Step 8. 12 Å 付近の重なりと誤差の確認

追加サンプリングの効果を定量的に見るため、(1) 各ウインドウのヒストグラム重なりと、
(2) 12 Å 付近の統計誤差 `df_i` を確認します。

In [ ]:
# 各ウインドウのヒストグラム (元=破線, 追加=実線) で重なりを確認
stops = [(0.00, "#3366CC"), (0.5, "#E64B35"), (1.00, "#52C569")]
cmap = LinearSegmentedColormap.from_list("brg", stops)

fig, ax = plt.subplots(figsize=(8, 4))
for k, wdw in enumerate(windows):
    z = z_kn[k, :N_k[k]]
    hist, bins = np.histogram(z, bins=12, density=True)
    ctr = 0.5 * (bins[:-1] + bins[1:])
    ls = "--" if wdw["kappa"] == KAPPA_ORIG else "-"
    lw = 1.0 if wdw["kappa"] == KAPPA_ORIG else 1.8
    ax.plot(ctr, hist, ls, lw=lw, color=cmap(k / len(windows)), alpha=0.8)
ax.axvspan(11.6, 12.4, color="#ffd166", alpha=0.15)
ax.set_xlim(10, 16)
ax.set_xlabel("Reaction coordinate z (A)")
ax.set_ylabel("Probability density")
ax.set_title("Window overlap (dashed=original k=2.5, solid=extra k=10)")
ax.grid(True)
plt.show()

# 12 A 付近の誤差
mask = (zc_new >= 11.6) & (zc_new <= 12.4)
print(f"12 A 付近 (11.6-12.4) の平均統計誤差 df: {df_new[mask].mean():.3f} kT")